# 6.5: Custom Layers

In [2]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## 6.5.1: Layers without Parameters

In [3]:
# Very similar to the module implementation from `Section 6.1`
# To build, we simply need to inherit from the base layer class and implement the forward propagation function
class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()

In [4]:
layer = CenteredLayer()
layer(torch.tensor([1.0, 2, 3, 4, 5]))

tensor([-2., -1.,  0.,  1.,  2.])

In [5]:
net = nn.Sequential(nn.LazyLinear(128), CenteredLayer())

c:\Users\Py Torch\.conda\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


In [ ]:
Y = net(torch.rand(4,8))
Y.mean() # small nonzero number due to quantization

tensor(3.9581e-09, grad_fn=<MeanBackward0>)

## 6.5.2: Layers with Parameters:

In [ ]:
class MyLinear(nn.Module):
    def __init__(self, in_units, units): # self, # input params, # output params
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))
        
    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return F.relu(linear) # Baked-in ReLU

In [8]:
linear = MyLinear(5,3)
linear.weight

Parameter containing:
tensor([[-0.9442, -0.3673,  0.3384],
        [-0.7607,  0.7092, -1.6315],
        [ 0.5906, -0.0083, -2.1216],
        [ 0.7505,  1.1686,  1.0040],
        [-0.6570,  1.6101,  0.4882]], requires_grad=True)

In [ ]:
linear(torch.rand(2,5)) # 2,5 x 5,3 -> 2,3

tensor([[0.0000, 0.0000, 0.0000],
        [0.5216, 0.0000, 0.0000]])

In [ ]:
net = nn.Sequential(MyLinear(64,8), MyLinear(8,1))
net(torch.rand(2,64)) # 2,64 x 64,8 x 8,1 -> 2x1

tensor([[0.0000],
        [6.2777]])

# Summary

We can design basic layers via the basic layer class. This allows us to define flexible new layers that behave differently from any existing layers in the library. Once defined, custom alyers can be invoked in arbitrary contexts and architectures. Layers can have local parameters, which can be created through build-in functions.